# Pick the TAPVid-MV evaluation episodes

`select_episodes.py` narrows 5000+ episodes down to a candidate pool it can
defend from the metrics alone: quality thresholds, a floor on how far the arm
travels, and an even spread over scenes. What it cannot see is whether an
episode is *interesting* — whether the manipulation is worth tracking, whether
two candidates from different scenes are doing the same thing anyway.

So it stops at a pool and this notebook picks the release set out of it by eye.

**How to use**

1. Run the cells top to bottom. The preview cell decodes every candidate once
   and caches it; that is the slow part, a few minutes, and only happens once.
2. Work through the picker: **Keep** puts the episode in the set, **Skip**
   drops it, **Back** undoes the last decision.
3. Run the last cell to write `episodes_eval50.txt`, which is what
   `run_export.sh` reads.

Each preview is one row per camera and eight frames across the episode, so a
frozen arm or a repeat of the previous scene is visible at a glance.

In [ ]:
import os, sys, csv, json
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import cv2
from IPython.display import display, clear_output
import ipywidgets as widgets

REPO = os.path.dirname(os.path.abspath("."))
if os.path.basename(os.getcwd()) != "tapvidmv":
    REPO = os.getcwd()
    HERE = os.path.join(REPO, "tapvidmv")
else:
    HERE = os.getcwd()
    REPO = os.path.dirname(HERE)
sys.path.insert(0, REPO)

import core.io

CANDIDATES  = os.path.join(HERE, "episodes_eval150_details.csv")
OUT_LIST    = os.path.join(HERE, "episodes_eval50.txt")
PREVIEW_DIR = os.path.join(HERE, "previews")
DEPTH_ROOT  = os.path.join(core.io.OUTPUT_ROOT, "depth")
N_TARGET    = 50

os.makedirs(PREVIEW_DIR, exist_ok=True)
with open(CANDIDATES) as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} candidates from {os.path.basename(CANDIDATES)}")
print(f"previews -> {PREVIEW_DIR}")
print(f"target    {N_TARGET} episodes -> {os.path.basename(OUT_LIST)}")

In [ ]:

def contact_sheet(ep, depth_root=None, n_cols=8, tile_w=200):
    depth_root = depth_root or DEPTH_ROOT
    ep_dir = os.path.join(depth_root, ep)
    if not os.path.isdir(ep_dir):
        return None
    cams = sorted(c for c in os.listdir(ep_dir)
                  if os.path.isdir(os.path.join(ep_dir, c)))
    strips = []
    for cam in cams:
        path = os.path.join(ep_dir, cam, "video_left.mp4")
        if not os.path.exists(path):
            continue
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        wanted = set(np.linspace(0, max(total - 1, 0), n_cols).astype(int).tolist())
        tiles, i = [], 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if i in wanted:
                h, w = frame.shape[:2]
                tiles.append(cv2.resize(frame, (tile_w, int(tile_w * h / w))))
            i += 1
        cap.release()
        if tiles:
            tiles += [tiles[-1]] * (n_cols - len(tiles))
            strips.append(np.hstack(tiles[:n_cols]))
    if not strips:
        return None
    width = min(s.shape[1] for s in strips)
    return np.vstack([s[:, :width] for s in strips])


def preview_path(ep):
    return os.path.join(PREVIEW_DIR, ep.replace("+", "_") + ".jpg")


def _build_one(ep):
    out = preview_path(ep)
    if os.path.exists(out):
        return ep, True
    sheet = contact_sheet(ep)
    if sheet is None:
        return ep, False
    cv2.imwrite(out, sheet, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return ep, True


todo = [r["episode_id"] for r in rows if not os.path.exists(preview_path(r["episode_id"]))]
print(f"{len(todo)} to build, {len(rows) - len(todo)} already cached")

failed = []
if todo:
    with ThreadPoolExecutor(max_workers=min(32, os.cpu_count() or 8)) as pool:
        futures = {pool.submit(_build_one, ep): ep for ep in todo}
        for n, fut in enumerate(as_completed(futures), 1):
            ep, ok = fut.result()
            if not ok:
                failed.append(ep)
            if n % 10 == 0 or n == len(todo):
                print(f"  {n}/{len(todo)}", end="\r")
print(f"\ndone. {len(failed)} episodes had no readable video" +
      (f": {failed[:3]}" if failed else ""))

CANDIDATE_ROWS = [r for r in rows if os.path.exists(preview_path(r["episode_id"]))]
print(f"{len(CANDIDATE_ROWS)} candidates ready to review")

## Pick

**Keep** adds the episode, **Skip** drops it, **Back** undoes the last
decision. The counter stops you at `N_TARGET`; nothing is written to disk
until the last cell.

In [ ]:
def _f(row, key, fmt="{:.3f}"):
    return fmt.format(float(row[key]))

def summarise(row):
    ep = row["episode_id"]
    return (f"{ep}\n"
            f"scene {ep.split('+')[1]}   site {row.get('site', '?')}   "
            f"{_f(row, 'n_frames', '{:.0f}')} frames   "
            f"{_f(row, 'n_cameras', '{:.0f}')} cameras\n"
            f"ee_travel {_f(row, 'ee_travel_m')} m   "
            f"joint_range {_f(row, 'joint_range_max_rad')} rad   "
            f"gripper {_f(row, 'gripper_range')}\n"
            f"chamfer {_f(row, 'chamfer_total', '{:.4f}')}   "
            f"depth_residual {_f(row, 'depth_residual_overall_median_mm', '{:.1f}')} mm   "
            f"static pts {_f(row, 'n_static', '{:.0f}')}")

state = {"i": 0, "kept": [], "history": []}

img_box  = widgets.Image(format="jpeg")
info     = widgets.HTML()
progress = widgets.HTML()
b_keep   = widgets.Button(description="Keep", button_style="success", icon="check")
b_skip   = widgets.Button(description="Skip", button_style="danger",  icon="times")
b_back   = widgets.Button(description="Back", icon="undo")
ui = widgets.VBox([progress, info,
                   widgets.HBox([b_keep, b_skip, b_back]), img_box])

def render():
    i, kept = state["i"], state["kept"]
    scenes = {e.split("+")[1] for e in kept}
    progress.value = (f"<b>kept {len(kept)}/{N_TARGET}</b> over {len(scenes)} scenes"
                      f" &nbsp;&nbsp; reviewed {i}/{len(CANDIDATE_ROWS)}")
    if i >= len(CANDIDATE_ROWS) or len(kept) >= N_TARGET:
        info.value = "<b>Done — run the next cell to save.</b>"
        img_box.value = b""
        for b in (b_keep, b_skip):
            b.disabled = True
        return
    for b in (b_keep, b_skip):
        b.disabled = False
    row = CANDIDATE_ROWS[i]
    dup = " &nbsp; <span style='color:#c00'>(scene already kept)</span>" \
          if row["episode_id"].split("+")[1] in scenes else ""
    info.value = "<pre style='margin:4px 0'>" + summarise(row) + "</pre>" + dup
    with open(preview_path(row["episode_id"]), "rb") as f:
        img_box.value = f.read()

def decide(keep):
    i = state["i"]
    if i >= len(CANDIDATE_ROWS):
        return
    ep = CANDIDATE_ROWS[i]["episode_id"]
    if keep:
        state["kept"].append(ep)
    state["history"].append((i, keep))
    state["i"] = i + 1
    render()

def go_back(_):
    if not state["history"]:
        return
    i, kept = state["history"].pop()
    if kept and state["kept"] and state["kept"][-1] == CANDIDATE_ROWS[i]["episode_id"]:
        state["kept"].pop()
    state["i"] = i
    render()

b_keep.on_click(lambda _: decide(True))
b_skip.on_click(lambda _: decide(False))
b_back.on_click(go_back)

render()
display(ui)

In [ ]:
kept = state["kept"]
scenes = {e.split("+")[1] for e in kept}
print(f"{len(kept)} episodes over {len(scenes)} scenes")

if not kept:
    print("Nothing kept — not writing.")
else:
    with open(OUT_LIST, "w") as f:
        for ep in sorted(kept):
            f.write(ep + "\n")
    print(f"wrote {OUT_LIST}")
    print("next:  bash tapvidmv/run_export.sh")